# 02. Model Training, Evaluation & Comparison
**Project:** Student Performance Classification  
**Lead:** Raghav (Core ML) | **UI Demo:** Aabiya | **Documentation/PPT:** Divyanshi  

### Objectives:
1. Perform stratified 80/20 train/test split.
2. Build Scikit-Learn Preprocessing Pipelines (Scaler + Imputer) to strictly prevent **Data Leakage**.
3. Train 3 suitable classical classification algorithms:
   - **Logistic Regression**
   - **Decision Tree Classifier**
   - **Random Forest Classifier**
4. Evaluate each model on test data using:
   - Accuracy, Precision (Macro), Recall (Macro), F1-Score (Macro)
   - Confusion Matrices
5. Export comparison table (`outputs/model_comparison.csv`) and serialize winning pipeline (`models/best_student_model.joblib`).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project root is in python path
sys.path.append(os.path.abspath('..'))

from src.data_preprocessing import load_dataset, clean_dataset, split_features_and_target, split_train_test
from src.train_models import build_model_pipelines, train_all_models
from src.evaluate import compare_and_save_results

## 1. Load Cleaned Dataset and Split
We separate features $X$ and target $y$, then perform an 80/20 stratified split.

In [ ]:
DATA_PATH = '../data/student_data.csv'
TARGET_COLUMN = 'Performance'  # Adjust once dataset arrives

try:
    df = load_dataset(DATA_PATH)
    df_clean = clean_dataset(df)
    X, y = split_features_and_target(df_clean, target_column=TARGET_COLUMN)
    X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2, random_state=42)
    print(f"Training instances: {len(X_train)}, Testing instances: {len(X_test)}")
except (FileNotFoundError, KeyError) as e:
    print(f"[INFO] Pending real dataset at {DATA_PATH}. Error info: {e}")

## 2. Build and Train Classical ML Pipelines
Each algorithm is wrapped in a Scikit-Learn Pipeline containing `SimpleImputer` and `StandardScaler`.

In [ ]:
if 'X_train' in locals():
    model_pipelines = build_model_pipelines(random_state=42)
    fitted_models = train_all_models(model_pipelines, X_train, y_train)

## 3. Comprehensive Evaluation, Comparison & Model Saving
We evaluate all three models on `X_test, y_test`, plot confusion matrices, and save the best model.

In [ ]:
if 'fitted_models' in locals():
    class_labels = list(np.unique(y_test))
    comparison_df, best_name, best_pipeline = compare_and_save_results(
        fitted_models=fitted_models,
        X_test=X_test,
        y_test=y_test,
        class_labels=class_labels,
        output_csv='../outputs/model_comparison.csv'
    )
    display(comparison_df)